# AIStor Tables with PyIceberg
Query Iceberg tables directly from MinIO's built-in catalog.

In [ ]:
import os
from pyiceberg.catalog import load_catalog

minio_endpoint = os.getenv('MINIO_ENDPOINT', 'http://localhost:9000')
access_key = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
secret_key = os.getenv('MINIO_SECRET_KEY', 'minioadmin')

# AIStor Tables exposes the Iceberg REST catalog at /_iceberg
catalog_uri = minio_endpoint.rstrip('/') + '/_iceberg'

catalog = load_catalog(
    'aistor',
    **{
        'type': 'rest',
        'uri': catalog_uri,
        's3.endpoint': minio_endpoint,
        's3.access-key-id': access_key,
        's3.secret-access-key': secret_key,
        's3.path-style-access': 'true',
    }
)
print(f'Connected to Iceberg catalog at: {catalog_uri}')

In [ ]:
namespaces = catalog.list_namespaces()
print(f'Namespaces ({len(namespaces)}):')
for ns in namespaces:
    print(f'  {".".join(ns)}')
    tables = catalog.list_tables(ns)
    for tbl in tables:
        print(f'    - {"." .join(tbl)}')

In [ ]:
# Load the first available table, or prompt user to specify
all_tables = []
for ns in namespaces:
    all_tables.extend(catalog.list_tables(ns))

if not all_tables:
    print('No Iceberg tables found in this catalog.')
    print('Create tables via the Trino SQL editor or the data generator first.')
else:
    table_id = all_tables[0]
    table = catalog.load_table(table_id)
    print(f'Loaded table: {"." .join(table_id)}')
    print(f'\nSchema:')
    for field in table.schema().fields:
        print(f'  {field.name}: {field.field_type}')

In [ ]:
if all_tables:
    scan = table.scan(limit=1000)
    arrow_table = scan.to_arrow()
    df = arrow_table.to_pandas()
    print(f'Read {len(df)} rows')
    display(df.head())
else:
    print('Skipped — no tables available.')

**Note:** This notebook requires an AIStor Tables-enabled MinIO instance.

AIStor Tables is MinIO's built-in Apache Iceberg catalog, available on enterprise MinIO deployments.
The catalog REST endpoint is automatically exposed at `<minio-endpoint>/_iceberg`.

To create Iceberg tables, use the **Trino** component in DemoForge with queries like:
```sql
CREATE TABLE iceberg.mydb.orders (
    id BIGINT,
    amount DOUBLE,
    created_at TIMESTAMP
) WITH (format = 'PARQUET');
```